In [1]:
from utils import *
from torch import dist  

def add_summary_behavior(
    sub_ids,
    features=None,
    *,
    on_missing_sub="skip",      # {"skip","raise"}
    on_missing_col="nan",       # {"nan","raise"}
    min_n=1,                    # minimum # valid rows for column-based features
):
    """
    Compute arbitrary per-subject features from behavior data.

    Parameters
    ----------
    sub_ids : iterable
        Subject IDs.
    features : dict[str, spec] or None
        Maps output_name -> spec, where spec can be:
          (A) callable: fn(df) -> scalar
          (B) ("col",  colname, stat_or_fn)
              - stat_or_fn: {"mean","median","sum","std","min","max"} or callable(series)->scalar
          (C) ("cols", [col1, col2, ...], fn)
              - fn: callable(df_subset)->scalar OR callable(*arrays)->scalar (see below)
          (D) ("coord", (x_col, y_col), fn)
              - fn(coords, df)->scalar where coords is (N,2) float array

        Notes on ("cols", ...):
          - If fn accepts 1 argument, we pass df_subset (numeric-coerced, rows with any NaN dropped).
          - If fn accepts k arguments, we pass k numpy arrays (one per column), aligned after dropping NaNs.

    on_missing_sub : {"skip","raise"}
        What to do if load_behavior fails/returns None/empty.
    on_missing_col : {"nan","raise"}
        What to do if a required column is missing for a feature.
    min_n : int
        Minimum number of valid (non-NaN) rows required for a feature; otherwise NaN.

    Returns
    -------
    pd.DataFrame
        Columns: sub_id + one column per feature key.
    """
    features = {} if features is None else dict(features)

    _STAT_FNS = {
        "mean":   lambda s: float(s.mean()),
        "median": lambda s: float(s.median()),
        "sum":    lambda s: float(s.sum()),
        "std":    lambda s: float(s.std()),
        "min":    lambda s: float(s.min()),
        "max":    lambda s: float(s.max()),
    }

    def _col_missing(msg):
        if on_missing_col == "raise":
            raise KeyError(msg)
        return np.nan

    def _coerce_series(df, col):
        if col not in df.columns:
            return None
        return pd.to_numeric(df[col], errors="coerce")

    def _eval_feature(df, spec):
        # (A) callable(df) -> scalar
        if callable(spec):
            return float(spec(df))

        if not (isinstance(spec, (tuple, list)) and len(spec) >= 1):
            raise ValueError(f"Invalid feature spec: {spec!r}")

        kind = spec[0]

        # (B) ("col", colname, stat_or_fn)
        if kind == "col":
            if len(spec) != 3:
                raise ValueError(f'("col", colname, stat_or_fn) expected; got {spec!r}')
            col, stat_or_fn = spec[1], spec[2]
            s = _coerce_series(df, col)
            if s is None:
                return _col_missing(f"Missing column: {col}")
            s = s.dropna()
            if len(s) < min_n:
                return np.nan

            if isinstance(stat_or_fn, str):
                key = stat_or_fn.lower()
                if key not in _STAT_FNS:
                    raise ValueError(f"Unknown stat {stat_or_fn!r}. Choose {sorted(_STAT_FNS)}.")
                return _STAT_FNS[key](s)

            if callable(stat_or_fn):
                return float(stat_or_fn(s))

            raise ValueError(f"stat_or_fn must be str or callable; got {type(stat_or_fn)}")

        # (C) ("cols", [col1,...], fn)
        if kind == "cols":
            if len(spec) != 3:
                raise ValueError(f'("cols", [cols...], fn) expected; got {spec!r}')
            cols, fn = list(spec[1]), spec[2]
            if any(c not in df.columns for c in cols):
                missing = [c for c in cols if c not in df.columns]
                return _col_missing(f"Missing columns: {missing}")

            d = df[cols].apply(pd.to_numeric, errors="coerce")
            d = d.dropna(axis=0, how="any")
            if len(d) < min_n:
                return np.nan

            # Decide whether to pass df_subset vs per-column arrays.
            try:
                n_params = fn.__code__.co_argcount
            except Exception:
                n_params = 1  # best-effort default

            if n_params <= 1:
                return float(fn(d))
            if n_params == len(cols):
                arrays = [d[c].to_numpy() for c in cols]
                return float(fn(*arrays))

            # Fallback: pass df_subset
            return float(fn(d))

        # (D) ("coord", (x_col,y_col), fn)
        if kind == "coord":
            if len(spec) != 3:
                raise ValueError(f'("coord", (x,y), fn) expected; got {spec!r}')
            (x_col, y_col), fn = tuple(spec[1]), spec[2]
            x = _coerce_series(df, x_col)
            y = _coerce_series(df, y_col)
            if x is None or y is None:
                return _col_missing(f"Missing coord cols: {(x_col, y_col)}")

            m = x.notna() & y.notna()
            if int(m.sum()) < max(min_n, 1):
                return np.nan
            coords = np.column_stack([x[m].to_numpy(), y[m].to_numpy()])
            return float(fn(coords, df))

        raise ValueError(f"Unknown feature kind: {kind!r}")

    rows = []
    for sub_id in sub_ids:
        try:
            df = load_behavior(sub_id)
        except Exception:
            if on_missing_sub == "raise":
                raise
            continue

        if df is None or len(df) == 0:
            if on_missing_sub == "raise":
                raise ValueError(f"Empty behavior for sub_id={sub_id}")
            continue

        row = {"sub_id": sub_id}
        for out_name, spec in features.items():
            try:
                row[out_name] = _eval_feature(df, spec)
            except Exception:
                # Robust default: feature-level failure becomes NaN
                row[out_name] = np.nan
        rows.append(row)

    return pd.DataFrame(rows)

def hull_volume(coords):
    return ConvexHull(coords).volume

def hull_end_volume(coords):
    coords_by_char = organize_by_character(coords)
    coords_by_char = np.array([char[-1] for char in coords_by_char])
    return ConvexHull(coords_by_char).volume

def rt_diffs(rts, eps=1e-6):
    rt_mask = np.isfinite(rts) & (rts > 0)
    rt = np.log(rts[rt_mask] + eps)
    dim = np.asarray(decision_trials["dimension"])[rt_mask]

    aff = rt[dim == "affil"]
    pow = rt[dim == "power"]
    if aff.size == 0 or pow.size == 0:
        return np.nan
    return float(np.mean(aff) - np.mean(pow))

def _apply_responded_mask(x, responded):
    """Return float array with non-responded trials set to NaN."""
    x = np.asarray(x, float)
    if responded is None:
        return x
    r = np.asarray(responded).astype(bool)
    if r.shape[0] != x.shape[0]:
        raise ValueError(f"responded length ({r.shape[0]}) != trials ({x.shape[0]})")
    out = x.copy()
    out[~r] = np.nan
    return out

def _per_character_summary(x, *, kind, trialwise_char_roles=None, neutrals=False):
    """
    Summarize trialwise 1D array per character.
    kind: "mean" or "end"
    Returns: array of length n_chars (float) with NaNs as needed.
    """
    by_char = organize_by_character(x, trialwise_char_roles=trialwise_char_roles, neutrals=neutrals)

    vals = []
    for xc in by_char:
        xc = np.asarray(xc, float)
        if kind == "mean":
            vals.append(np.nanmean(xc))
        elif kind == "end":
            # last finite within that character; if none, NaN
            m = np.isfinite(xc)
            vals.append(xc[np.where(m)[0][-1]] if m.any() else np.nan)
        else:
            raise ValueError("kind must be 'mean' or 'end'")
    return np.asarray(vals, float)

def social_dist(
    dists,
    *,
    trials="char_end",                # {"all_avg","char_avg","char_end"}
    responded=None,
    trialwise_char_roles=None,
    neutrals=False,
):
    """
    Social distance summary with responded-trial exclusion.

    trials:
      - "all_avg"  : nanmean across all trials
      - "char_avg" : nanmean within character, then nanmean across characters
      - "char_end" : last finite within character, then nanmean across characters
    """
    dists = _apply_responded_mask(dists, responded=responded)

    trials = str(trials).lower()
    if trials in {"all", "all_avg", "avg_all", "trial_avg"}:
        return float(np.nanmean(dists))

    if trials in {"char_avg", "avg"}:
        per_char = _per_character_summary(
            dists, kind="mean", trialwise_char_roles=trialwise_char_roles, neutrals=neutrals
        )
        return float(np.nanmean(per_char))

    if trials in {"char_end", "end"}:
        per_char = _per_character_summary(
            dists, kind="end", trialwise_char_roles=trialwise_char_roles, neutrals=neutrals
        )
        return float(np.nanmean(per_char))

    raise ValueError("trials must be one of {'all_avg','char_avg','char_end'}")

def dim_corr(
    coords,
    df=None,
    *,
    trials="all",                     # {"all","char_avg","char_end"}
    responded=None,
    trialwise_char_roles=None,
    neutrals=False,
):
    """
    Affiliation/power correlation with responded-trial exclusion.

    trials:
      - "all"      : pearsonr across all (trialwise) coords
      - "char_avg" : per-character mean coords, then pearsonr across characters
      - "char_end" : per-character end coords,  then pearsonr across characters
    """
    coords = np.asarray(coords, float)
    if coords.ndim != 2 or coords.shape[1] != 2:
        raise ValueError(f"coords must be (T,2). Got {coords.shape}")

    # Prefer explicit responded; else fall back to df["responded"] if present
    if responded is None and df is not None and ("responded" in df.columns):
        responded = df["responded"].to_numpy()

    responded = None if responded is None else np.asarray(responded).astype(bool)
    if responded is not None and responded.shape[0] != coords.shape[0]:
        raise ValueError(f"responded length ({responded.shape[0]}) != trials ({coords.shape[0]})")

    trials = str(trials).lower()

    if trials == "all":
        X = coords.copy()
        if responded is not None:
            X[~responded, :] = np.nan
        a = X[:, 0]
        p = X[:, 1]
        m = np.isfinite(a) & np.isfinite(p)
        if m.sum() < 3:
            return np.nan
        r, _ = scipy.stats.pearsonr(a[m], p[m])
        return float(r)

    if trials in {"char_avg", "avg"}:
        a = _apply_responded_mask(coords[:, 0], responded)
        p = _apply_responded_mask(coords[:, 1], responded)
        a_c = _per_character_summary(a, kind="mean", trialwise_char_roles=trialwise_char_roles, neutrals=neutrals)
        p_c = _per_character_summary(p, kind="mean", trialwise_char_roles=trialwise_char_roles, neutrals=neutrals)
        m = np.isfinite(a_c) & np.isfinite(p_c)
        if m.sum() < 3:
            return np.nan
        r, _ = scipy.stats.pearsonr(a_c[m], p_c[m])
        return float(r)

    if trials in {"char_end", "end"}:
        a = _apply_responded_mask(coords[:, 0], responded)
        p = _apply_responded_mask(coords[:, 1], responded)
        a_c = _per_character_summary(a, kind="end", trialwise_char_roles=trialwise_char_roles, neutrals=neutrals)
        p_c = _per_character_summary(p, kind="end", trialwise_char_roles=trialwise_char_roles, neutrals=neutrals)
        m = np.isfinite(a_c) & np.isfinite(p_c)
        if m.sum() < 3:
            return np.nan
        r, _ = scipy.stats.pearsonr(a_c[m], p_c[m])
        return float(r)

    raise ValueError("trials must be one of {'all','char_avg','char_end'}")


In [13]:
# load in so they don't have any code-mediated changes
data  = pd.read_excel(f'{data_dir}/data.xlsx')
data_tavares = pd.read_excel(f'{data_dir}/other-samples/tavares/data.xlsx')
data_online = pd.read_excel(f'{data_dir}/other-samples/online/data.xlsx')

features = {
    # # QC
    # "response_rate": lambda df: df["responded"].mean(),

    # # reaction-times
    # "rt_log_mean": lambda df: np.nanmean(np.log(df.loc[df["responded"], "reaction_time"])),
    # "rt_diff": lambda df: rt_diffs(df['reaction_time'].to_numpy()),

    # # per-character end state then average across characters
    # "distance_end": lambda df: social_dist(
    #     df["distance"].to_numpy(),
    #     trials="char_end",
    #     responded=df["responded"].to_numpy(),
    # ),
    # "dim_corr_end": ("coord", ("affil_coord", "power_coord"),
    #     lambda coords, df: dim_corr(coords, df, trials="char_end")
    # ),

    # # per-character average then average across characters
    # "distance_avg": lambda df: social_dist(
    #     df["distance"].to_numpy(),
    #     trials="char_avg",
    #     responded=df["responded"].to_numpy(),
    # ),

    # # average across *all* responded trials (if you want it)
    # "distance_all_avg": lambda df: social_dist(
    #     df["distance"].to_numpy(),
    #     trials="all_avg",
    #     responded=df["responded"].to_numpy(),
    # ),
    # "dim_corr_all": ("coord", ("affil_coord", "power_coord"),
    #     lambda coords, df: dim_corr(coords, df, trials="all")
    # ),

    "volume": lambda df: hull_volume(df[["affil_coord", "power_coord"]].to_numpy()),
}

df = add_summary_behavior(incl_subs, features)
df = df.merge(data, on="sub_id", how="right")

df_tav = add_summary_behavior(incl_subs_tavares, features)
df_tav = df_tav.merge(data_tavares, on="sub_id", how="inner")

# df_online = add_summary_behavior(incl_subs_online, features)
# df_online = df_online.merge(data_online, on="sub_id", how="inner")

In [18]:
df.to_excel(f'{data_dir}/data.xlsx', index=False)

In [19]:
df_tav.to_excel(f'{data_dir}/other-samples/tavares/data.xlsx', index=False)